## Test Simple Sampling - sampler

In [14]:
if False:
    import os
    os.environ["QUANTDATA_RAW_DATA"] = "!/quantdata/raw" # better set these to environment variables
    os.environ["QUANTDATA_TRAINING_DATA"] = "!/quantdata/training" # better set these to environment variables
    os.environ["QUANTDATA_PROCESSING_DATA"] = "!/quantdata/processing" # better set these to environment variables
    os.environ["QUANTDATA_CACHE_MODE"] = "off" # turns off global cache mode
import importlib as imp

import quantdata as _
imp.reload(_)
import quantdata.biz_calendar as _
imp.reload(_)
import quantdata.widb as _
imp.reload(_)
import quantdata.widb.client as _
imp.reload(_)
import quantdata.widb.day as _
imp.reload(_)
import quantdata.widb.sampler as _
imp.reload(_)
import quantdata.dysanos as _
imp.reload(_)
import quantdata.dysanos.data as _
imp.reload(_)
import quantdata.dysanos.data.simple as _
imp.reload(_)

from quantdata.dysanos.data.simple import SimpleSampler, sample_simple_days, SimpleConfig, SamplingConfig, ExpiryFilter, RawDayFilter, SampleDaysConfig, DateRange, PoolConfig, verbose_sample_days
from cdxcore.verbose import Context
from IPython.display import display
from datetime import date

def test():
    simple_config = SimpleConfig(
            tickers = ['SPX','NDX','RUT'],
            date_range = DateRange(start=date(2025, 1, 1), end=date(2025, 7, 1)),
            raw_day_filter = RawDayFilter(),
            expiry_filter = ExpiryFilter(),
            sample_days_config = SampleDaysConfig(),
            sampling_config = SamplingConfig(num_samples=20)
    )
    pool_config = PoolConfig(num_workers=4, threading=True) # no parallel processing for testing
    verbose = Context.all
    refresh_cache = False

    sampler = SimpleSampler(simple_config=simple_config, pool_config=pool_config, refresh_cache=refresh_cache, verbose=verbose)
    return sampler

sampler = test()
print("/done")

00: Generating samples for 3 tickers NDX, RUT and SPX with from 2025-01-01 to 2025-07-01... done; this took 0.045s.
00: Found the following number of samples per ticker: NDX: 122, SPX: 122 and RUT: 122.
00: Found date range 2025-01-02 to 2025-07-01 with 122 total dates. First validation date for 25 steps per sample is 2025-05-06.
00: Using up to 1000 options per date.
01:   Ticker NDX: using 12 training samples (accross 83 training dates), and 3 validation samples (accross 39 validation dates).
01:   Ticker SPX: using 12 training samples (accross 83 training dates), and 3 validation samples (accross 39 validation dates).
01:   Ticker RUT: using 12 training samples (accross 83 training dates), and 3 validation samples (accross 39 validation dates).
00: Created 36 training samples and 9 validation samples based on the date split at 2025-05-06. That implies 20% of all samples are validation samples.
/done


## Test sampling

In [15]:
"""
Shapes
"""


from prettytable import PrettyTable

def test_shapes():
    shapes = sampler.shapes( 10 )

    tbl = PrettyTable()
    tbl.field_names = ["Ticker", "Shape"]

    for k, v in shapes.items():
        tbl.add_row([k, v])
    
    print(tbl)

test_shapes()   



+-------------------+----------------+
|       Ticker      |     Shape      |
+-------------------+----------------+
|   ask_pure_delta  | (10, 25, 1000) |
|    ask_pure_dk    | (10, 25, 1000) |
|   ask_pure_gamma  | (10, 25, 1000) |
|   ask_pure_price  | (10, 25, 1000) |
|   ask_pure_theta  | (10, 25, 1000) |
|   ask_pure_vega   | (10, 25, 1000) |
|      ask_vol      | (10, 25, 1000) |
|   bid_pure_delta  | (10, 25, 1000) |
|    bid_pure_dk    | (10, 25, 1000) |
|   bid_pure_gamma  | (10, 25, 1000) |
|   bid_pure_price  | (10, 25, 1000) |
|   bid_pure_theta  | (10, 25, 1000) |
|   bid_pure_vega   | (10, 25, 1000) |
|      bid_vol      | (10, 25, 1000) |
|      call_ask     | (10, 25, 1000) |
|      call_bid     | (10, 25, 1000) |
|         df        | (10, 25, 1000) |
|        fwd        | (10, 25, 1000) |
|       is_am       | (10, 25, 1000) |
|    is_american    | (10, 25, 1000) |
|      is_valid     | (10, 25, 1000) |
| log_open_interest | (10, 25, 1000) |
|     log_volume    | (10

In [ ]:
from prettytable import PrettyTable
from cdxcore.util import fmt_list, fmt_big_byte_number

def test_batch():
    print("Loading torch...", end='', flush=True)
    import torch as torch
    print("done.")
    g = torch.Generator(device="cpu")
    g.manual_seed(1234)
    batch_ix, batch = sampler(g, trn=True, batch_size=10, device="cpu", return_batch_indices=True)

    # indices
    tbl = PrettyTable()
    tbl.field_names = ["Ticker", "Date"]

    for bix in batch_ix:
        date = sampler.data[bix.ticker].dates[bix.index]
        tbl.add_row([bix.ticker, bix.date])
    print("Samples")
    print(tbl)

    tbl = PrettyTable()
    tbl.field_names = ["Ticker", "Shape", "Type", "Size", "Device"]

    for k, v in batch.items():
        tbl.add_row([k, tuple(v.shape), v.dtype, fmt_big_byte_number(v.nbytes), v.device])
    
    print(tbl)

test_batch()



Loading torch...done.
Samples
+--------+------------+
| Ticker |    Date    |
+--------+------------+
|  RUT   | 2025-03-31 |
|  NDX   | 2025-01-24 |
|  RUT   | 2025-03-17 |
|  RUT   | 2025-01-16 |
|  SPX   | 2025-01-07 |
|  NDX   | 2025-03-03 |
|  NDX   | 2025-02-07 |
|  RUT   | 2025-03-03 |
|  SPX   | 2025-02-07 |
|  NDX   | 2025-02-24 |
+--------+------------+
+-------------------+----------------+---------------+------------+--------+
|       Ticker      |     Shape      |      Type     |    Size    | Device |
+-------------------+----------------+---------------+------------+--------+
|   ask_pure_delta  | (10, 25, 1000) | torch.float32 |  976.56KB  |  cpu   |
|    ask_pure_dk    | (10, 25, 1000) | torch.float32 |  976.56KB  |  cpu   |
|   ask_pure_gamma  | (10, 25, 1000) | torch.float32 |  976.56KB  |  cpu   |
|   ask_pure_price  | (10, 25, 1000) | torch.float32 |  976.56KB  |  cpu   |
|   ask_pure_theta  | (10, 25, 1000) | torch.float32 |  976.56KB  |  cpu   |
|   ask_pure_vega 